In [ ]:
#We have various types of chains in langchain.
#1. Simple Chain: A simple chain is a sequence of steps that are executed in order. Each step can be a function, a prompt, or another chain. The output of one step can be used as the input for the next step.
#2. Sequential Chain: A sequential chain is a chain that executes a series of steps in sequence. Each step is executed one after the other, and the output of one step is passed as input to the next step.
#3. Parallel Chain: A parallel chain is a chain that executes multiple steps in parallel. Each step is executed simultaneously, and the outputs of all steps are collected together at the end.
#4. Conditional Chain: A conditional chain is a chain that executes different steps based on certain conditions coming as output from last step. The chain can branch out into different paths depending on the input or the output of previous steps.
#5. Loop Chain: A loop chain is a chain that repeats a set of steps until a certain condition is met. The chain can loop back to previous steps or continue to the next steps based on the condition.

1

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

d:\Gen AI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
load_dotenv()

# model = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0, max_tokens=1000)
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite', temperature=0, max_tokens=2000)

parser = StrOutputParser()

In [3]:
#1. Simple Chain 
prompt = PromptTemplate(
    template='Generate 3 interesting facts about {topic}',
    input_variables=['topic']
)

chain = prompt | model | parser   #simple chain

result = chain.invoke({'topic':'cricket'})

print(result)

chain.get_graph().print_ascii()

ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

In [9]:
# 2. Sequential Chain
prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Summarize the following report in 3 bullet points considering the key points of {report} for the discussed topic of report: {report}',
    input_variables=['report']
)

chain = prompt1 | model | parser | prompt2 | model | parser   #sequential chain

result = chain.invoke({'topic':'Govt. Job Vs Private Job in India'})

print(result)

Based on the provided text, which is very limited, here are the key points:

*   The report's central theme is a detailed comparison between government and private sector jobs in India.
*   It addresses this choice as a significant and ongoing dilemma ("perennial dilemma") for a vast number of job aspirants in the country.
*   The report aims to provide insights into this complex decision-making process for millions of individuals.


In [16]:
# 3. Parallel Chain
from langchain_core.runnables import RunnableParallel

prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)


prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)


prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parallel_chain = RunnableParallel(
    {
        'notes': prompt1 | model | parser,
        'quiz': prompt2 | model | parser
    }
)

merge_chain = prompt3 | model | parser

chain = parallel_chain | merge_chain

text = '''
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
'''

result = chain.invoke({'text': text})
print(result)

# chain.get_graph().print_ascii()

Here is a single document combining the notes and quiz on Support Vector Machines (SVMs).

---

# Support Vector Machines (SVMs): Notes and Quiz

This document provides a concise overview of


In [6]:
# 4. Conditional Chain
#importing runnables for conditional chain.
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

# To parse output
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

class SentimentOutput(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

# sentiment_parser = PydanticOutputParser(pydantic_object=SentimentOutput)

structured_model = model.with_structured_output(SentimentOutput)

# Define the prompt to analyze sentiment
sentiment_prompt = PromptTemplate(
    template='Analyze the sentiment of the following feedback and classify it as positive or negative: {feedback}',
    input_variables=['feedback'],
    # partial_variables={'format_instructions': sentiment_parser.get_format_instructions()}
)

classification_chain = sentiment_prompt | structured_model

# Define the prompt for positive feedback
positive_prompt = PromptTemplate(
    template='The feedback is positive. Generate a max 2 sample thank you response for the following feedback: {feedback}',
    input_variables=['feedback']
)

# Define the prompt for negative feedback
negative_prompt = PromptTemplate(
    template='The feedback is negative. Generate a max 2 sample for apology response for the following feedback: {feedback}',
    input_variables=['feedback']
)


# Define the branching via conditional chain
branch_chain = RunnableBranch(
    (lambda x:x["sentiment"].sentiment == 'positive', positive_prompt | model | parser),
    (lambda x:x["sentiment"].sentiment == 'negative', negative_prompt | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

# chain = classification_chain | branch_chain
chain = (
    RunnableParallel(
        sentiment=classification_chain, 
        feedback=lambda x: x["feedback"]
    ) 
    | branch_chain
)
feedback = "This is a beautiful phone!"
result = chain.invoke({"feedback": feedback})

print(result)


Here are two sample thank you responses for the feedback "This is a beautiful phone!":

**Option 1 (Short & Sweet):**

> Thank you so much! We're thrilled you think so.

**Option 2 (Slightly More Enthusiastic):**

> That's wonderful to hear! We're so glad you're enjoying the beautiful design. Thank you for your kind words!
